# Kurulum ve Veri Okuma

In [8]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, countDistinct, from_unixtime, to_date, hour, dayofweek, min as spark_min, max as spark_max, avg as spark_avg
from datetime import datetime
import pandas as pd

# Spark oturumu
spark = SparkSession.builder.appName("EDA_Notebook").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Silver tablosunu oku
df = spark.read.parquet("/tmp/parquet/silver_reviews")

print("✅ Spark oturumu başlatıldı ve Silver verisi yüklendi.")

✅ Spark oturumu başlatıldı ve Silver verisi yüklendi.


# Temel İstatistikler ve Şema

In [9]:
print("📊 1. TEMEL İSTATİSTİKLER")
print("-" * 30)

toplam_satir = df.count()
print(f"• Toplam satır sayısı: {toplam_satir:,}")

print(f"\n• Veri Şeması:")
df.printSchema()

# İlk 5 kaydı Pandas formatında şık bir tablo olarak gör
print("\n• İlk 5 Kayıt Önizleme:")
df.limit(5).toPandas()

📊 1. TEMEL İSTATİSTİKLER
------------------------------
• Toplam satır sayısı: 15,365

• Veri Şeması:
root
 |-- timestamp: double (nullable = true)
 |-- kullanici_ID: string (nullable = true)
 |-- olay_tipi: string (nullable = true)
 |-- ilgili_ID: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- star_rating: double (nullable = true)
 |-- review_body: string (nullable = true)


• İlk 5 Kayıt Önizleme:


,timestamp,kullanici_ID,olay_tipi,ilgili_ID,kategori,star_rating,review_body
0,1.778088e+09,8464339,review_posted,B00L108SAW,Electronics,4.0,Here’s the quickie...<br /><br />1. Cool LCD s...
1,1.778088e+09,23768147,review_posted,B00LM0U8I6,Electronics,5.0,"Good build quality, LOVE the low profile fit."
2,1.778088e+09,2155598,review_posted,B002MYQTEI,Electronics,5.0,I honestly love this product I've had it over ...
3,1.778088e+09,10195727,review_posted,B006U3O566,Electronics,5.0,good connectors for your speakers to receiver ...
4,1.778088e+09,3004043,review_posted,B00YC3BZP0,Electronics,5.0,Great product


# Benzersiz Değerler ve Eksik Veri Analizi

In [10]:
print("📈 2. BENZERSİZ DEĞER VE EKSİK VERİ ANALİZİ")
print("-" * 30)

# Benzersiz değerler
unique_stats = df.select(
    countDistinct("kullanici_ID").alias("Kullanıcı"),
    countDistinct("ilgili_ID").alias("Ürün"),
    countDistinct("kategori").alias("Kategori"),
    countDistinct("olay_tipi").alias("Olay")
).toPandas()

print("Benzersiz Değer Sayıları:")
display(unique_stats)

# Eksik değerler
print("\nEksik (Null) Değer Kontrolü:")
null_counts = df.select([count(col(c)).alias(c) for c in df.columns]).collect()[0]
for col_name in df.columns:
    missing = toplam_satir - null_counts[col_name]
    status = "Tamam ✓" if missing == 0 else f"{missing} adet eksik!"
    print(f"- {col_name}: {status}")

📈 2. BENZERSİZ DEĞER VE EKSİK VERİ ANALİZİ
------------------------------
Benzersiz Değer Sayıları:


,Kullanıcı,Ürün,Kategori,Olay
0,8813,6220,1,1



Eksik (Null) Değer Kontrolü:
- timestamp: Tamam ✓
- kullanici_ID: Tamam ✓
- olay_tipi: Tamam ✓
- ilgili_ID: Tamam ✓
- kategori: Tamam ✓
- star_rating: Tamam ✓
- review_body: Tamam ✓


# Kategorik Dağılım

In [11]:
print("📋 3. KATEGORİK DEĞİŞKEN DAĞILIMI")
print("-" * 30)

print("Kategori Dağılımı:")
df.groupBy("kategori").agg(count("*").alias("Toplam_Sayı")).orderBy(col("Toplam_Sayı").desc()).show(truncate=False)

print("Olay Tipi Dağılımı:")
df.groupBy("olay_tipi").agg(count("*").alias("Toplam_Sayı")).show(truncate=False)

📋 3. KATEGORİK DEĞİŞKEN DAĞILIMI
------------------------------
Kategori Dağılımı:
+-----------+-----------+
|kategori   |Toplam_Sayı|
+-----------+-----------+
|Electronics|15365      |
+-----------+-----------+

Olay Tipi Dağılımı:
+-------------+-----------+
|olay_tipi    |Toplam_Sayı|
+-------------+-----------+
|review_posted|15365      |
+-------------+-----------+



# Zaman Serisi ve Trend Analizi

In [12]:
print("⏰ 4. ZAMAN SERİSİ ANALİZİ")
print("-" * 30)

# Zaman kolonlarını ekle
df_time = df.withColumn("date", to_date(from_unixtime("timestamp"))) \
            .withColumn("hour", hour(from_unixtime("timestamp")))

# Günlük trend
print("Son Günlerin Yorum Sayıları:")
df_time.groupBy("date").count().orderBy("date").show()

# Saatlik dağılım (Pandas ile daha iyi görünür)
hourly_stats = df_time.groupBy("hour").count().orderBy("hour").toPandas()
print("Saatlik Mesaj Yoğunluğu:")
display(hourly_stats)

# Zaman aralığı detayları
stats = df.select(spark_min("timestamp"), spark_max("timestamp")).collect()[0]
print(f"\n• En eski veri: {datetime.fromtimestamp(stats[0])}")
print(f"• En yeni veri: {datetime.fromtimestamp(stats[1])}")

⏰ 4. ZAMAN SERİSİ ANALİZİ
------------------------------
Son Günlerin Yorum Sayıları:
+----------+-----+
|      date|count|
+----------+-----+
|2026-05-06|15365|
+----------+-----+

Saatlik Mesaj Yoğunluğu:


,hour,count
0,17,7195
1,18,8170



• En eski veri: 2026-05-06 17:16:39.976044
• En yeni veri: 2026-05-06 18:07:06.739931
